# 프로토콜 v1.2 — WS1: 개발 데이터 학습 곡선, PI-BDP와 GB (탐색 분석, CPU)

**시험 데이터를 쓰지 않습니다.** 고장까지 관측된 개발 엔진 55대만 씁니다(v1.2 2.1).

- **바깥 고리:** 서브셋마다 고장 엔진을 하나씩 평가용으로 뺍니다.
- **학습 크기 m:** 남은 고장 엔진 중 m ∈ {2, 3, 5}대(남은 수보다 작은 값만)와 "전체"
- **추출:** 조합이 3개 이하면 모두, 많으면 고정 시드(2026)로 3개를 뽑습니다. 합계 415회입니다.
  - 추출 목록(`ws1_draws.csv`)은 이 노트북이 만들어 저장하고, GPU 노트북(TSMixer·TCN·S-pool)이 같은 목록을 해시로 확인해 씁니다.
- **모델:** PI-BDP(v1.0 V0, noWeibull, melding α = 0.5, α = 1; 같은 표본에 가중만 다름)와 GB(v1.0 같은 정보 기준선, 고른 설정)
  - 모두 학습 m대로만 적합합니다.
  - m이 작을 때의 처리(대각 공분산, k = 1, Weibull 실패 시 재가중 없음)는 v1.2 2.3절 규칙대로이며, 쓰인 경우를 기록합니다.
- **채점 (v1.2 2.4):**
  - 주 채점: CruiseBench 순항 창(공개 마스크로 사이클별 창 수 계산), η = 5% 상한, 사이클 t의 창에 t − 1 출력 − 1, 첫 사이클 창 제외. 모든 모델에 같은 규칙입니다.
  - 보조 채점: 사이클 단위, 상한 없음
  - PI-BDP 보정 전 구간의 포함률, 폭, interval score, pinball도 계산합니다.
- 평가 엔진마다 추출 평균을 낸 엔진별 점수(`ws1_engine_scores.csv`)를 저장합니다. TSMixer·TCN과의 붓스트랩 비교는 GPU 실행 뒤 채점 노트북에서 합니다.

**실행 전:** 이 노트북을 커밋한 주소를 `RUN_COMMIT_URL`에 넣으세요. **런타임:** CPU, 고용량 RAM. 끊기면 다시 실행하면 끝난 서브셋은 건너뜁니다.

In [ ]:
import os, sys, json, glob, time, hashlib, re, shutil, subprocess
import numpy as np, pandas as pd
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = '/content/drive/MyDrive/PI-BDP_NCMAPSS'; LIB = '/content/ws1lib_dir'; CBDIR = '/content/CruiseBench'
else:
    ROOT = os.environ.get('NCM_ROOT', './ncm'); LIB = os.environ.get('WS1_LIB', './ws1lib_dir'); CBDIR = os.environ.get('CB_DIR', './CruiseBench')
OUT = f'{ROOT}/addendum_v12/ws1'; os.makedirs(f'{OUT}/raw', exist_ok=True); os.makedirs(LIB, exist_ok=True)
URL = r'https://github\.com/.+/commit/[0-9a-f]{7,40}$'
ADDENDUM_V12_URL = 'https://github.com/goniworld/pibdp-ncmapss/commit/f01eecf82f5bc45ed9a90d44d8361b8dcb217d63'
RUN_COMMIT_URL = ''     # 이 노트북을 커밋한 주소
assert re.match(URL, ADDENDUM_V12_URL) and re.match(URL, RUN_COMMIT_URL), '커밋 주소를 넣으세요.'
SUB9 = ['DS01-005', 'DS02-006', 'DS03-012', 'DS04', 'DS05', 'DS06', 'DS07', 'DS08a-009', 'DS08c-008']
SUBSETS = os.environ.get('WS1_SUBSETS', ','.join(SUB9)).split(',')          # 로컬 시험용 (Colab에서는 9개)
CAPS = dict(zip(SUB9, [38, 34, 34, 43, 38, 41, 39, 34, 28]))
FAILED_EXPECTED = dict(zip(SUB9, [4, 6, 9, 4, 5, 6, 6, 9, 6]))               # v1.2 2.1
N_SAMP = int(os.environ.get('WS1_NSAMP', 2000))
def sha(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 22), b''): h.update(b)
    return h.hexdigest()
print('CPU 코어', os.cpu_count())

In [ ]:
# === 라이브러리: v1.0 고정본 + WS1 코드 ===
LIB1_SRC = '"""WS1 learning curves on development data (protocol addendum v1.2, section 2): PI-BDP variants and GB."""\nimport itertools\nimport numpy as np\nimport pandas as pd\nfrom scipy.stats import gaussian_kde\nfrom sklearn.ensemble import HistGradientBoostingRegressor\nimport pibdp_ncm as P\n\nQC = [f\'q{int(round(p * 1000)):03d}\' for p in P.QUANTS]\nVARIANTS = [\'V0\', \'noWeibull\', \'meld0.5\', \'meld1\']\nGRID = np.arange(0, 400.5, 0.5)\n\n\ndef draws_table(failed, seed=2026):\n    """failed: {subset: sorted list of failed dev units}. One row per (subset, held-out, m, draw)."""\n    rows = []\n    for si, (s, units) in enumerate(failed.items()):\n        for x in units:\n            rest = [u for u in units if u != x]\n            for m in [2, 3, 5, \'all\']:\n                if m == \'all\':\n                    combos = [tuple(rest)]\n                else:\n                    if m >= len(rest): continue\n                    allc = list(itertools.combinations(rest, m))\n                    if len(allc) <= 3:\n                        combos = allc\n                    else:\n                        rng = np.random.default_rng([seed, si, int(x), m])\n                        combos = [allc[i] for i in sorted(rng.choice(len(allc), 3, replace=False))]\n                for d, cmb in enumerate(combos):\n                    rows.append(dict(subset=s, heldout=int(x), m=str(m), draw=d, n_train=len(cmb), train=\' \'.join(map(str, cmb))))\n    return pd.DataFrame(rows)\n\n\ndef fit_pibdp(D, train, cfg):\n    """v1.0 PI-BDP fitted on the training engines only, with the v1.2 2.3 small-m rules."""\n    notes = []\n    m = P.PIBDP(**cfg).fit(D, train)\n    if m.gpa.k_eff == 2 and np.min(np.bincount(m.gpa.labels, minlength=2)) < 2:\n        m = P.PIBDP(**dict(cfg, k=1)).fit(D, train); notes.append(\'k=1 (a mode had <2 engines)\')\n    if len(train) <= 3:\n        d = m.deg; Pm = d.P_u\n        d.mu = np.median(Pm, 0); d.S = np.diag(np.var(Pm, 0, ddof=1)) + 1e-6 * np.eye(3)\n        d.Linv = np.linalg.cholesky(np.linalg.inv(d.S)).T\n        notes.append(\'diagonal covariance\')\n    if not (np.isfinite(m.wb.k) and np.isfinite(m.wb.lam)):\n        m.wb = None; notes.append(\'weibull failed -> noWeibull\')\n    return m, \';\'.join(notes)\n\n\ndef log_pi0(deg, rng, n=20000, kde_n=5000):\n    th = rng.multivariate_normal(deg.mu, deg.S, n, method=\'svd\'); D = rng.normal(deg.D_mu, deg.D_sd, n)\n    a, b, c = th[:, 0], np.exp(th[:, 1]), np.exp(th[:, 2]); arg = (D - a) / b\n    T = np.where(arg > 1, np.log(np.maximum(arg, 1)) / c, np.nan) * P.TS; T = T[np.isfinite(T) & (T > 0)]\n    if len(T) < 50: return None\n    return np.log(np.maximum(gaussian_kde(T[:kde_n])(GRID), 1e-300))\n\n\ndef predict_variants(m, x, n=2000, seed=0):\n    """Same sampling as PIBDP.predict_unit (rng seed + j); four re-weightings of the same samples."""\n    t_all, y_all = m.hi_unit(x)\n    lp0 = log_pi0(m.deg, np.random.default_rng(seed + 10_000))\n    rows = []\n    for j in range(len(t_all)):\n        rng = np.random.default_rng(seed + j)\n        r = m.deg.rul_samples(t_all[:j + 1], y_all[:j + 1], n, rng); Tn = t_all[j] + r\n        W = {\'noWeibull\': np.ones_like(r)}\n        if m.wb is not None:\n            lw = m.wb.logpdf(Tn); W[\'V0\'] = np.exp(lw - lw.max())\n            if lp0 is not None:\n                lr = lw - np.interp(Tn, GRID, lp0)\n                for a in (0.5, 1.0):\n                    z = a * lr; W[f\'meld{a:g}\'] = np.exp(z - z.max())\n        row = dict(cycle=int(t_all[j]))\n        for v in VARIANTS:\n            w = W.get(v, W[\'noWeibull\'])          # Weibull or pi0 unavailable -> no re-weighting (recorded in note)\n            q = P.wquantile(r, w / w.sum(), P.QUANTS)\n            row.update({f\'{v}_{c}\': val for c, val in zip(QC, q)})\n        rows.append(row)\n    return pd.DataFrame(rows), lp0 is not None\n\n\ndef gb_predict(m, train, x, eol, params, seed):\n    Xtr, ytr, _ = P.fold_xy(m.Z, train, eol)\n    gb = HistGradientBoostingRegressor(random_state=seed, early_stopping=False, **params).fit(Xtr, ytr)\n    Xte, _, idx = P.fold_xy(m.Z, [x], eol)\n    return pd.DataFrame({\'cycle\': idx.cycle.values, \'GB\': gb.predict(Xte)})\n\n\ndef run_job(D, job, cfg, gb_params, n=2000):\n    train = [int(u) for u in job[\'train\'].split()]; x = int(job[\'heldout\'])\n    m, note = fit_pibdp(D, train, cfg)\n    pv, pi0_ok = predict_variants(m, x, n=n)\n    if not pi0_ok: note = (note + \';pi0 failed -> meld=noWeibull\').strip(\';\')\n    g = gb_predict(m, train, x, D[\'eol\'], gb_params, seed=int(job[\'draw\']) + 1)\n    out = pv.merge(g, on=\'cycle\', how=\'left\')\n    out.insert(0, \'true\', D[\'eol\'][x] - out.cycle)\n    for k in [\'subset\', \'heldout\', \'m\', \'draw\']: out.insert(0, k, job[k])\n    out[\'note\'] = note\n    return out\n'
MAN = json.load(open(f'{ROOT}/freeze_v1/MANIFEST.json'))
assert sha(f'{ROOT}/freeze_v1/pibdp_ncm.py') == MAN['library_sha256'], '라이브러리가 v1.0 고정본과 다릅니다.'
shutil.copy(f'{ROOT}/freeze_v1/pibdp_ncm.py', f'{LIB}/pibdp_ncm.py'); open(f'{LIB}/ws1lib.py', 'w').write(LIB1_SRC)
sys.path.insert(0, LIB); os.environ['PYTHONPATH'] = LIB + os.pathsep + os.environ.get('PYTHONPATH', '')
import importlib, pibdp_ncm as P, ws1lib as L; importlib.reload(P); importlib.reload(L)
cfg = json.load(open(f'{ROOT}/phase1/config.json')); GB_PARAMS = json.load(open(f'{ROOT}/phase2/ml_selected.json'))['GB']['params']
json.dump(dict(pibdp_ncm=MAN['library_sha256'], ws1lib=hashlib.sha256(LIB1_SRC.encode()).hexdigest()), open(f'{OUT}/ws1_code_sha256.json', 'w'), indent=1)
def cfg_for(s): return dict(baseline=cfg['baseline'], n_h=cfg['n_h'], k=cfg['k'][s], shrink=cfg['shrink'])
print('라이브러리 = v1.0 고정본 확인 | GB', GB_PARAMS)

In [ ]:
# === CruiseBench 순항 마스크로 개발 엔진의 사이클별 창 수 (창 256, 간격 10) ===
if not os.path.exists(CBDIR):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/NostalgiaJohn/CruiseBench', CBDIR], check=True)
subprocess.run(['git', '-C', CBDIR, 'checkout', '-q', 'ef26d40'], check=True)
MASK = os.environ.get('CB_MASK', f'{CBDIR}/dataset/cruise_mask/common_altitude_cruise_mask.h5')
import h5py
with h5py.File(MASK, 'r') as h: R = h['cycles'][()]
WIN = pd.DataFrame({'subset': np.char.decode(R['dataset']), 'split': np.char.decode(R['split']), 'unit': R['unit'], 'cycle': R['cycle'],
                    'detected': R['detected'].astype(bool), 'L': R['end_sample'] - R['start_sample'] + 1})
WIN = WIN[(WIN.split == 'dev') & WIN.detected & (WIN.L >= 256)].assign(n_win=lambda d: (d.L - 256) // 10 + 1)[['subset', 'unit', 'cycle', 'n_win']]
WIN.to_csv(f'{OUT}/ws1_dev_windows.csv', index=False)
print('마스크', sha(MASK)[:12], '| 개발 순항 창', int(WIN.n_win.sum()))

In [ ]:
# === 고장 엔진과 추출 목록 (GPU 노트북과 공유) ===
rec0 = json.load(open(f'{ROOT}/phase0/sha256_phase0.json')); FAILED = {}
for s in SUBSETS:
    f = f'{ROOT}/phase0/dev_{s}.npz'; assert sha(f) == rec0[f'dev_{s}.npz'], f'{s}: 개발 파일 해시 불일치'
    d = np.load(f); A = d['A']; an = list(d['A_var']); u = A[:, an.index('unit')].astype(int); cy = A[:, an.index('cycle')].astype(int)
    eol = pd.Series(cy).groupby(u).max(); FAILED[s] = sorted(int(x) for x in eol.index[eol < P.CAP_CYCLES]); d.close()
print({s: len(v) for s, v in FAILED.items()})
if SUBSETS == SUB9:
    assert {s: len(v) for s, v in FAILED.items()} == FAILED_EXPECTED, '고장 엔진 수가 v1.2 2.1과 다름'
DR = L.draws_table(FAILED)
if SUBSETS == SUB9: assert len(DR) == 415, f'추출 수 {len(DR)} ≠ 415 (v1.2 2.6)'
DF = f'{OUT}/ws1_draws.csv'
if os.path.exists(DF): assert pd.read_csv(DF, dtype={'m': str}).equals(DR.astype({'m': str})), '저장된 추출 목록과 다름'
else: DR.to_csv(DF, index=False)
print('추출', len(DR), '| 해시', sha(DF)[:12]); print(DR.groupby(['subset', 'm']).size().unstack(fill_value=0).to_string())

In [ ]:
# === 학습과 예측 (서브셋마다 저장, 끊기면 이어서) ===
from joblib import Parallel, delayed
t0 = time.time()
for s in SUBSETS:
    f = f'{OUT}/raw/{s}.csv.gz'
    if os.path.exists(f): continue
    D = P.load_dev(f'{ROOT}/phase0/dev_{s}.npz'); jobs = DR[DR.subset == s].to_dict('records'); ts = time.time()
    res = Parallel(n_jobs=-1)(delayed(L.run_job)(D, j, cfg_for(s), GB_PARAMS, N_SAMP) for j in jobs)
    pd.concat(res, ignore_index=True).to_csv(f, index=False)
    print(f'{s}: {len(jobs)}회 {(time.time() - ts) / 60:.1f}분 (누적 {(time.time() - t0) / 60:.1f}분)')

In [ ]:
# === 채점: 순항 창 (주), 사이클 단위 (보조) ===
QL = P.QUANTS; QC = L.QC; MODELS = L.VARIANTS + ['GB']
def pinball(Q, y):
    d = y[:, None] - Q; tau = QL[None, :]
    return np.mean(np.maximum(tau * d, (tau - 1) * d), axis=1)
recs = []
for s in SUBSETS:
    Rw = pd.read_csv(f'{OUT}/raw/{s}.csv.gz', dtype={'m': str}); cap = CAPS[s]
    for (x, m, d), g in Rw.groupby(['heldout', 'm', 'draw']):
        g = g.sort_values('cycle'); eol = int((g.true + g.cycle).iloc[0])
        w = WIN[(WIN.subset == s) & (WIN.unit == x) & (WIN.cycle <= eol)].sort_values('cycle')
        w = w[w.cycle > w.cycle.min()]                                   # 창이 있는 첫 사이클 제외
        prev = g.assign(cycle=g.cycle + 1)                               # t - 1 출력을 t로
        X = w.merge(prev, on='cycle', how='inner'); assert len(X) == len(w), f'{s} {x}: t-1 예측 없는 사이클'
        n = X.n_win.values.astype(float); y = np.minimum(eol - X.cycle.values, cap).astype(float)
        rec = dict(subset=s, heldout=x, m=m, draw=d, n=n.sum(), note=g.note.iloc[0] if 'note' in g else '')
        for mdl in MODELS:
            p = np.clip((X[f'{mdl}_q500'] if mdl != 'GB' else X['GB']).values - 1, 0, cap)
            rec[f'{mdl}_sse'] = float(np.sum(n * (p - y) ** 2))
            # 사이클 단위 (상한 없음, 모든 사이클, 같은 사이클의 출력)
            pc = (g[f'{mdl}_q500'] if mdl != 'GB' else g['GB']).values
            rec[f'{mdl}_cyc_sse'] = float(np.sum((pc - g.true.values) ** 2)); rec['n_cyc'] = len(g)
        for v in L.VARIANTS:
            Q = np.clip(X[[f'{v}_{q}' for q in QC]].values - 1, 0, cap); Lo, Up = Q[:, 1], Q[:, 7]
            rec[f'{v}_cover'] = float(np.sum(n * ((y >= Lo) & (y <= Up)))); rec[f'{v}_width'] = float(np.sum(n * (Up - Lo)))
            rec[f'{v}_iscore'] = float(np.sum(n * ((Up - Lo) + 20 * (Lo - y) * (y < Lo) + 20 * (y - Up) * (y > Up))))
            rec[f'{v}_pinball'] = float(np.sum(n * pinball(Q, y)))
        recs.append(rec)
DRAWS = pd.DataFrame(recs); DRAWS.to_csv(f'{OUT}/ws1_draw_scores.csv', index=False)
num = [c for c in DRAWS.columns if c not in ('subset', 'heldout', 'm', 'draw', 'note')]
ENG = DRAWS.groupby(['subset', 'heldout', 'm'])[num].mean().reset_index()       # 평가 엔진마다 추출 평균 (v1.2 2.5)
ENG.to_csv(f'{OUT}/ws1_engine_scores.csv', index=False)
print('추출', len(DRAWS), '| 평가 엔진 × m', len(ENG))

In [ ]:
# === 학습 곡선 (기술 통계; 붓스트랩은 GPU 결과와 함께 채점 노트북에서) ===
M_ORDER = ['2', '3', '5', 'all']
def curve(E):
    rows = []
    for m, g in E.groupby('m'):
        r = dict(m=m, n_engines=len(g))
        for mdl in MODELS:
            r[f'{mdl}_rmse'] = np.sqrt(g[f'{mdl}_sse'].sum() / g.n.sum()); r[f'{mdl}_cyc_rmse'] = np.sqrt(g[f'{mdl}_cyc_sse'].sum() / g.n_cyc.sum())
        for v in L.VARIANTS:
            for k in ['cover', 'width', 'iscore', 'pinball']: r[f'{v}_{k}'] = g[f'{v}_{k}'].sum() / g.n.sum()
        rows.append(r)
    return pd.DataFrame(rows).set_index('m').reindex([m for m in M_ORDER if m in set(E.m)])
CUR = curve(ENG); CUR.to_csv(f'{OUT}/ws1_curve_cpu.csv')
CURS = pd.concat({s: curve(ENG[ENG.subset == s]) for s in SUBSETS}, names=['subset']); CURS.to_csv(f'{OUT}/ws1_curve_cpu_subset.csv')
NOTES = DRAWS.assign(note=DRAWS.note.fillna('')).groupby('m').agg(
    diag=('note', lambda v: v.str.contains('diagonal').mean()), k1=('note', lambda v: v.str.contains('k=1').mean()),
    wb_fail=('note', lambda v: v.str.contains('weibull').mean()), pi0_fail=('note', lambda v: v.str.contains('pi0').mean())).reindex(CUR.index)
NOTES.to_csv(f'{OUT}/ws1_fallbacks.csv')
show = CUR[[f'{m}_rmse' for m in MODELS] + [f'{m}_cyc_rmse' for m in ['V0', 'GB']] + ['V0_cover', 'meld1_cover', 'noWeibull_cover', 'n_engines']]
print(show.round(3).to_string()); print(NOTES.round(3).to_string())
txt = '\n'.join([f"Addendum v1.2 WS1 CPU part (exploratory, dev data only) {time.strftime('%Y-%m-%d %H:%M:%S')}  addendum={ADDENDUM_V12_URL}  run={RUN_COMMIT_URL}",
    f"Failed dev engines {sum(len(v) for v in FAILED.values())}; draws {len(DR)} (ws1_draws.csv sha256 {sha(DF)}); samples {N_SAMP}",
    '', 'Learning curve (cruise windows, eta=5% caps, t-1 rule; RMSE pooled over held-out engines, draw-averaged):', CUR.round(3).to_string(),
    '', 'Per subset:', CURS[[f'{m}_rmse' for m in MODELS]].round(3).to_string(),
    '', 'Small-m fallbacks (share of draws):', NOTES.round(3).to_string()])
open(f'{OUT}/ws1_cpu_summary.txt', 'w').write(txt)
H = {os.path.relpath(p, OUT): sha(p) for p in sorted(glob.glob(f'{OUT}/**/*.*', recursive=True)) if not p.endswith('sha256_ws1_cpu.json')}
json.dump(H, open(f'{OUT}/sha256_ws1_cpu.json', 'w'), indent=1); print(txt)